# Inspect FINK schema

Look for latest Json file with naming convention all_schema_"API version"_YYYYMMDD  based on YYYYMMDD.

Currrent: 
PORTAL : "https://api.lsst.fink-portal.org"
API version : "/api/v1/"

Dump for : "statistics", "tags", "conesearch", "objects"
            "sources", "fp", "cutouts"

Find a pattern in the variable and display the endpoint/variable with the assocaited documentation

In [ ]:
import pandas as pd

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org"
API_V = "/api/v1/"

In [ ]:
api_version = API_V.replace("/", "_").strip("_")

In [ ]:
import os
import glob
from datetime import datetime

# 1. Chemin du dossier où se trouvent tes fichiers (modifie selon ton cas)
dossier = "./"  # Par défaut : dossier courant

# 2. Liste tous les fichiers correspondant au pattern
fichiers = glob.glob(os.path.join(dossier, f"all_schema_{api_version}_*.json"))

# 3. Si aucun fichier trouvé, affiche un message
FILE_OK = False
if not fichiers:
    print(f"Aucun fichier all_schema_{api_version}_*.json trouvé dans le dossier.")
else:
    # 4. Extrait la date de chaque fichier et trie par date
    fichiers_par_date = []
    for fichier in fichiers:
        # Extrait la date du nom de fichier (ex: 20260403)
        date_str = fichier.split(f"_{api_version}_")[1].split(".json")[0]
        date = datetime.strptime(date_str, "%Y%m%d")
        fichiers_par_date.append((date, fichier))

    # 5. Trie par date décroissante (le plus récent en premier)
    fichiers_par_date.sort(reverse=True, key=lambda x: x[0])

    # 6. Récupère le chemin du fichier le plus récent
    dernier_fichier = fichiers_par_date[0][1]

    # 7. Charge le fichier JSON
    import json
    with open(dernier_fichier, "r") as f:
        all_schema = json.load(f)

    # 8. Affiche le résultat
    print(f"Fichier chargé : {dernier_fichier}")
    FILE_OK = True

In [ ]:
df = pd.DataFrame(list(all_schema.items()), columns=["endpoint/var", "doc"])

In [ ]:
def print_df(df,nrows=10):
    ## Limite à N lignes AVANT de styliser
    if nrows is not None:
        df_limited = df.head(nrows)
    else:
        df_limited = df
        
    
    # Applique le style
    styled_df = df_limited.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('text-align', 'center')]
    }])
    
    # Affiche le DataFrame stylisé et limité
    display(HTML(styled_df.to_html()))

In [ ]:
print_df(df)

In [ ]:
def search(df, tag):
    # Filtrer les lignes où "endpoint/var" contient tag (insensible à la casse)
    filtered_df = df[df["endpoint/var"].str.contains(tag, 
                                                     case=False, regex=False)]
    return filtered_df

# Exemple:
search for variable with tag "detec"
|     | endpoint/var          | doc                                                                                                                                  |
|----:|:----------------------|:-------------------------------------------------------------------------------------------------------------------------------------|
|  31 | tags/r:detector       | Id of the detector where this diaSource was measured. Datatype short instead of byte because of DB concerns about unsigned bytes.    |
| 163 | conesearch/r:detector | Id of the detector where this diaSource was measured. Datatype short instead of byte because of DB concerns about unsigned bytes.    |
| 468 | sources/r:detector    | Id of the detector where this diaSource was measured. Datatype short instead of byte because of DB concerns about unsigned bytes.    |
| 593 | fp/r:detector         | Id of the detector where this forcedSource was measured. Datatype short instead of byte because of DB concerns about unsigned bytes. |

In [ ]:
print_df(search(df, tag="detec"),nrows=None)